# OpenAI Responses via Agent Endpoint — `@azure/ai-projects`

This notebook demonstrates creating a hosted agent, session, and calling the OpenAI Responses API via the agent endpoint using the `AIProjectClient`.

It mirrors the [`openaiWithAgentEndpointBasic.ts`](./openaiWithAgentEndpointBasic.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_AGENT_CONTAINER_IMAGE`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  HostedAgentDefinition,
  ProtocolVersionRecord,
  VersionRefIndicator,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const image = process.env["FOUNDRY_AGENT_CONTAINER_IMAGE"] ?? "<agent image>";
const agentName = "MySessionHostedAgent";
console.log(`Image: ${image}`);
console.log(`Agent name: ${agentName}`);

Image: crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest
Agent name: MySessionHostedAgent
Agent name: MySessionHostedAgent


In [2]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Create a hosted agent version
console.log("Creating agent...");
const agent = await project.agents.createVersion(
  agentName,
  {
    kind: "hosted",
    cpu: "0.5",
    memory: "1Gi",
    container_configuration: { image: image },
    protocol_versions: [{ protocol: "responses", version: "1.0.0" } as ProtocolVersionRecord],
  } as HostedAgentDefinition,
  {
    metadata: { enableVnextExperience: "true" },
  },
);
console.log(`Agent created (name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (name: MySessionHostedAgent, version: 7)


In [4]:
// Poll until agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, agent.version);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1})`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: creating (attempt 1)
Agent version status: creating (attempt 2)
Agent version status: creating (attempt 3)
Agent version status: creating (attempt 4)
Agent version status: active (attempt 5)


In [5]:
// Create a session
const versionIndicator: VersionRefIndicator = {
  type: "version_ref",
  agent_version: agent.version,
};
const session = await project.agents.createSession(agentName, versionIndicator);
console.log(`Session created (id: ${session.agent_session_id}, status: ${session.status})`);

Session created (id: 64359f1d8cd2177400SgTAkb43oMy8K99esFL8qKX7H0pzSStz, status: active)


In [6]:
// Create an OpenAI client bound to the agent endpoint
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient({
  azureConfig: { allowPreview: true, agentName: agentName },
});

In [7]:
// Call Responses API bound to the agent session
console.log("\nGenerating response...");
const response = await openAIClient.responses.create(
  {
    input: "What is the size of France in square miles?",
  },
  {
    body: { agent_session_id: session.agent_session_id },
  },
);
console.log(`Response output: ${response.output_text}`);


Generating response...
Response output: Echo: What is the size of France in square miles?


In [8]:
// Cleanup
console.log("\nCleaning up resources...");

await project.agents.deleteSession(agentName, session.agent_session_id);
console.log(`Session with id: ${session.agent_session_id} deleted.`);

await project.agents.deleteVersion(agentName, agent.version);
console.log(`Agent version ${agent.version} deleted.`);


Cleaning up resources...
Session with id: 64359f1d8cd2177400SgTAkb43oMy8K99esFL8qKX7H0pzSStz deleted.
Agent version 7 deleted.
